In [ ]:
import pandas as pd
import glob
import os

# 1. Path to the main directory
path = '/Users/daniel/PhD/Projects/psd-paths/outputs/specparam/' 

# Use recursive=True and ** to look into all subfolders
all_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

li = []

for filename in all_files:
    # Read the individual CSV
    df = pd.read_csv(filename, index_col=None, header=0)
    
    # 2. Extract subject ID from the filename
    # Based on your image: 'sub-101-specparam.csv' -> split by '-' and take first two parts
    file_basename = os.path.basename(filename)
    parts = file_basename.split('-')
    subject_id = f"{parts[0]}-{parts[1]}" # Results in 'sub-101'
    
    # 3. Add the subject column at the beginning
    df.insert(0, 'subject_id', subject_id)
    
    li.append(df)

# 4. Concatenate and Save
if li:
    combined_df = pd.concat(li, axis=0, ignore_index=True)
    combined_df.to_csv("all_subjects_specparam.csv", index=False)
    print(f"Successfully combined {len(li)} files.")
else:
    print("No CSV files found. Check your path or folder permissions.")

In [ ]:
combined_df 

In [ ]:
# 1. Define the columns to drop (peak-specific parameters)
cols_to_drop = ['CF', 'PW', 'BW']

# 2. Drop the columns and remove duplicate rows
# We use errors='ignore' just in case a file was missing one of these columns
df_unique = combined_df.drop(columns=cols_to_drop, errors='ignore').drop_duplicates()

# 3. Reset index for a clean dataframe
df_unique = df_unique.reset_index(drop=True)

# 4. Display results
print(f"Original row count (all peaks): {len(combined_df)}")
print(f"Unique observations (aperiodic/fit only): {len(df_unique)}")

# Save the unique observations to a new file
df_unique.to_csv("unique_specparam_results.csv", index=False)

# Preview the unique data (typically shows Offset, Exponent, Error, and R2)
df_unique.head()

In [ ]:
import pandas as pd

# 1. Start with df_unique (one row per channel per subject)
# We calculate a boolean for "Good Fit" (R2 >= 0.9)
df_unique['is_good_fit'] = df_unique['gof_rsquared'] >= 0.9

# 2. Group by subject to calculate the ratio of good channels
# The mean of a boolean column gives the percentage (e.g., 0.6 = 60% good channels)
subject_summary = df_unique.groupby('subject_id')['is_good_fit'].agg(['mean', 'count']).reset_index()
subject_summary.columns = ['subject_id', 'ratio_good_channels', 'total_channels']
subject_summary['ratio_good_channels'] = subject_summary['ratio_good_channels'].round(2)
# 3. Apply the exclusion criteria: 
# Exclude if ratio_good_channels < 0.5 (meaning more than 50% are bad fits)
subject_summary['status'] = subject_summary['ratio_good_channels'].apply(
    lambda x: 'Keep' if x >= 0.5 else 'Exclude'
)

# 4. Create the final separate tables
df_exclusion_report = subject_summary.copy()
excluded_subjects = subject_summary[subject_summary['status'] == 'Exclude']
final_kept_subjects = subject_summary[subject_summary['status'] == 'Keep']

# 5. Filter the actual data to get your clean dataset
df_final_clean = df_unique[df_unique['subject_id'].isin(final_kept_subjects['subject_id'])]

# --- Display the results ---
print(f"Exclusion Summary:")
print(f"- Total participants processed: {len(subject_summary)}")
print(f"- Participants excluded: {len(excluded_subjects)}")
print(f"- Participants remaining: {len(final_kept_subjects)}")

# Save the exclusion report for your records
df_exclusion_report.to_csv("participant_exclusion_report.csv", index=False)

# Preview the exclusion report table
df_exclusion_report.sort_values(by='ratio_good_channels')

Similarly to @mckeown2023TestretestReliabilitySpectral, we  excluded participants with specparam model fits < 0.9 R2 on more than 50% of the channels. Given the above criterium 4 subjects were identified as those to be exluded from analysis: `102 103 126 151`. 
<!-- (Check 139?) -->

## Results
I manually checked data:

- really  good quality 101, 105, 107,112 , 113,114, 115, 122, 123, 127, 128 , 131, 134, 135, 136, 137, 139 , 142, 143, 144, 149, 152
- good 108, 116, 124, 125, 138 , 145, 146, 148

- medium quality 104, 106, 109 , 110 , 111, 118, 119, 120, 121, 129, 132, 140, 147, 150

good 101, 108, 107, 111, 112, 113
medium 106, 104, 105, , 110, 